In [1]:
import pandas as pd
import polars as pl
import numpy as np
import os
from pathlib import Path
import pandas as pd
import re, pathlib

In [2]:
# =============================================================================
# 1.  Directory layout – pathlib all the way
# =============================================================================
SCRIPT_DIR   = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
PROJECT_ROOT = SCRIPT_DIR.parent          # edit if your notebook is elsewhere

DATA_DIR       = PROJECT_ROOT / "data/"
SIMULATION_DIR = DATA_DIR / "simulations/"          # folder with ATTRIBUTE_* and wide SSP file
TORNADO_SIM_DIR = SIMULATION_DIR /"data_for_LSU"
OUTPUT_DIR     = DATA_DIR / "output/"

In [3]:
louisiana = pd.read_csv(TORNADO_SIM_DIR / "louisiana.csv")

In [ ]:
louisiana

In [5]:
import pandas as pd

# 1) Filter your base‐case scenario
base_case = louisiana[louisiana['primary_id'] == 0].copy()

# 2) Detect the vehicle‐km column
dist_cols = [
    c for c in base_case.columns
    if c.startswith('vehicle_distance_traveled_trns_road_light')
]
if not dist_cols:
    raise KeyError("No vehicle_distance_traveled_trns_road_light_* column found")
dist_col = dist_cols[0]
vehicle_distance = base_case[dist_col]  # units: vkm

# 3) Define the three fuel‐mix fraction columns
relevant_fuels = ['diesel', 'electricity', 'gasoline']
frac_cols = {
    fuel: f'frac_trns_fuelmix_road_light_{fuel}'
    for fuel in relevant_fuels
}
for col in frac_cols.values():
    if col not in base_case.columns:
        raise KeyError(f"Missing fraction column: {col}")

# 4) Record the period-0 (baseline) shares
frac_baseline = {
    fuel: base_case[col].iloc[0]
    for fuel, col in frac_cols.items()
}

# 5) Compute how many vkm have switched fuels since the baseline
switched_from_diesel    = vehicle_distance * (frac_baseline['diesel']    - base_case[frac_cols['diesel']])
switched_from_gasoline  = vehicle_distance * (frac_baseline['gasoline']  - base_case[frac_cols['gasoline']])
switched_to_electricity = vehicle_distance * (base_case[frac_cols['electricity']] - frac_baseline['electricity'])

# 6) Use your lookup‐table multipliers (in $ per vkm)
#    * note the NEGATIVE sign for the electrification “cost” from the table
multipliers = {
    'electricity': {'cost': 0.039, 'saving':  0.012},
    'diesel':      {'cost':  0.000, 'saving':  0.000},
    'gasoline':    {'cost':  0.000, 'saving':  0.000},
}

# 7) Accumulate total cost & total savings
cost_series   = pd.Series(0.0, index=base_case.index)
saving_series = pd.Series(0.0, index=base_case.index)

for fuel in relevant_fuels:
    frac = base_case[frac_cols[fuel]]
    vkm  = vehicle_distance * frac
    cost_series   += vkm * multipliers[fuel]['cost']
    saving_series += vkm * multipliers[fuel]['saving']

# 8) Build the output table
output_fs = pd.DataFrame({
    'primary_id':                         base_case['primary_id'],
    'region':                             base_case['region'],
    'time_period':                        base_case['time_period'],
    'vehicle_distance_traveled_total_vkm': vehicle_distance,
    'switched_from_diesel_vkm':           switched_from_diesel,
    'switched_from_gasoline_vkm':         switched_from_gasoline,
    'switched_to_electricity_vkm':        switched_to_electricity,
    'fuel_switch_cost_$':                 cost_series,
    'fuel_switch_savings_$':              saving_series,
    'fuel_switch_net_cost_$':             cost_series - saving_series,
}, index=base_case.index)




In [ ]:
# # ——————————————————————————————————————————
# # 2) Grab the light‐duty vehicle‐km column
# # ——————————————————————————————————————————
# dist_cols = [c for c in base_case.columns
#              if c.startswith("vehicle_distance_traveled_trns_road_light")]
# if not dist_cols:
#     raise KeyError("No vehicle_distance_traveled_trns_road_light_* column found")
# vehicle_distance = base_case[dist_cols[0]]  # series of total vkm

# # ——————————————————————————————————————————
# # 3) Define the fuel‐mix fraction columns & baseline
# # ——————————————————————————————————————————
# relevant_fuels = ["diesel", "electricity", "gasoline"]
# frac_cols = {
#     f: f"frac_trns_fuelmix_road_light_{f}"
#     for f in relevant_fuels
# }
# # baseline shares at period 0
# frac_baseline = {
#     f: base_case[frac_cols[f]].iloc[0]
#     for f in relevant_fuels
# }

# # ——————————————————————————————————————————
# # 4) Compute switched‐to‐electricity vkm
# # ——————————————————————————————————————————
# switched_to_elec = (
#     vehicle_distance *
#     (base_case[frac_cols["electricity"]] - frac_baseline["electricity"])
# )

# # ——————————————————————————————————————————
# # 5) Apply your multipliers ONLY to the switched‐to‐electricity
# # ——————————————————————————————————————————
# # dollars per vkm
# elec_mult = {"cost":   0.039,    # cost per vkm
#              "saving": 0.012}    # saving per vkm

# cost_series   = switched_to_elec * elec_mult["cost"]
# saving_series = switched_to_elec * elec_mult["saving"]
# net_series    = cost_series - saving_series  # this is your capex

# # ——————————————————————————————————————————
# # 6) Build the final output table
# # ——————————————————————————————————————————
# output_fs = pd.DataFrame({
#     "primary_id":                           base_case["primary_id"],
#     "region":                               base_case["region"],
#     "time_period":                          base_case["time_period"],
#     "vehicle_distance_traveled_total_vkm":  vehicle_distance,
#     "switched_to_electricity_vkm":          switched_to_elec,
#     "fuel_switch_net_cost_$":               net_series,   # renamed capex
# })





In [6]:
output_fs = (
    output_fs

        .rename(columns={'fuel_switch_net_cost_$': 'capex'})             # rename to capex
)

In [7]:
output_fs

,primary_id,region,time_period,vehicle_distance_traveled_total_vkm,switched_from_diesel_vkm,switched_from_gasoline_vkm,switched_to_electricity_vkm,fuel_switch_cost_$,fuel_switch_savings_$,capex
0,0,louisiana,0,8.603325e+10,0.000000e+00,0.000000e+00,0.000000e+00,7.529756e+05,2.316848e+05,5.212908e+05
1,0,louisiana,1,8.132988e+10,-1.068198e+08,1.132746e+08,6.454665e+06,9.635429e+05,2.964748e+05,6.670682e+05
2,0,louisiana,2,8.321870e+10,-1.498349e+08,1.670442e+08,1.720921e+07,1.399501e+06,4.306158e+05,9.688856e+05
3,0,louisiana,3,8.757311e+10,-1.744021e+08,2.101610e+08,3.575882e+07,2.161047e+06,6.649374e+05,1.496109e+06
4,0,louisiana,4,8.833723e+10,-2.428882e+08,3.049185e+08,6.203023e+07,3.192319e+06,9.822520e+05,2.210067e+06
5,0,louisiana,5,9.943955e+10,-2.734146e+08,3.432410e+08,6.982625e+07,3.593533e+06,1.105703e+06,2.487831e+06
6,0,louisiana,6,9.610184e+10,-2.642374e+08,3.317200e+08,6.748252e+07,3.472915e+06,1.068589e+06,2.404326e+06
7,0,louisiana,7,9.870512e+10,-2.713953e+08,3.407059e+08,6.931054e+07,3.566993e+06,1.097536e+06,2.469456e+06
8,0,louisiana,8,1.016400e+11,-2.794649e+08,3.508364e+08,7.137140e+07,3.673053e+06,1.130170e+06,2.542883e+06
9,0,louisiana,9,1.049100e+11,-2.884559e+08,3.621236e+08,7.366759e+07,3.791224e+06,1.166530e+06,2.624693e+06


In [8]:
# 9) (Optional) Save as CSV
output_fs.to_csv(OUTPUT_DIR/'transportion_light_duty_fuel_switch_cost.csv', index=False)

print("Done. Columns in output:")
print(output_fs.columns.tolist())

Done. Columns in output:
['primary_id', 'region', 'time_period', 'vehicle_distance_traveled_total_vkm', 'switched_from_diesel_vkm', 'switched_from_gasoline_vkm', 'switched_to_electricity_vkm', 'fuel_switch_cost_$', 'fuel_switch_savings_$', 'capex']
